# 🌊 Ocean Modelling with ROMS: Install, Run and Analyse

### This is notebook 2 of 2

| | Notebook | What you do | Result |
|---|---|---|---|
| **1** | `Grid_IC_BC_Preparation_Steps.ipynb` | create and edit the grid, then initial, boundary and surface-forcing files | `roms_grd.nc`, `roms_ini.nc`, `roms_frc.nc` in Google Drive → `MyDrive/ROMS_class/inputs` |
| **2** | `ROMS_class_2026.ipynb` (this one) | compile and run ROMS with **your** files, then analyse | model output and plots |

> Run notebook 1 first, including its last cell (save to Google Drive). This notebook then copies your files from Drive.
> No files yet? Set `INPUTS = 'tutorial'` in Step 1 to run with the ready-made tutorial files instead.

**A hands-on class notebook (Google Colab).** In about two hours you will:

1. **Install** ROMS and its libraries on a fresh Colab machine
2. **Inspect the inputs**: grid, initial conditions, atmospheric forcing
3. **Configure** the model: physics switches (`io_1deg.h`) and run parameters (`roms_io_1deg.in`)
4. **Compile** ROMS
5. **Run** a 30-day simulation of the Indian Ocean at 1° resolution
6. **Save** the output before Colab recycles the machine
7. **Analyse** the output: SST, currents, vertical sections, mixed layer, surface heat budget, and a comparison with GLORYS

| Step | What happens | Time on Colab |
|---|---|---|
| 1 Install | apt packages, download inputs, clone ROMS | ~2 min |
| 4 Compile | `build_roms.sh -j 2` | ~2–3 min |
| 5 Run | 30 model days | see note in Step 5 |
| 7 Analyse | plots | interactive |

> **Model set-up at a glance**: Indian Ocean, about 1° (100 km) resolution; grid size and levels come from **your** grid file;
> all four boundaries are **closed walls**, so no boundary file is read; initial state from GLORYS (16 Jan 1993); ERA5 monthly-mean forcing applied through bulk formulae.

Run the cells **from top to bottom**. Every cell is safe to re-run.

## Step 1 — Install ROMS and download the inputs

We install:
* **NetCDF-Fortran**, the file format ROMS reads and writes
* **OpenMPI**, because ROMS is built here as an MPI program (`romsM`)
* **netcdf-bin**, which gives us `ncdump` for inspecting files

We also **pin** both downloads to fixed versions so that everyone in the room builds the same code.

In [ ]:
%%shell
set -e
echo "Installing libraries (≈1 min)..."
apt-get -qq update > /dev/null
apt-get -qq install -y gfortran libnetcdff-dev netcdf-bin libopenmpi-dev openmpi-bin > /dev/null 2>&1
echo "  nf-config: $(nf-config --version)   mpif90: $(mpif90 --version | head -1)"
pip install -q cartopy > /dev/null 2>&1 && echo "  cartopy installed"

cd /content
# Tutorial inputs (pinned commit)
wget -q -nc https://raw.githubusercontent.com/Rohithocean/physical_oceanography/c0ea486bfc73ec088b7dd8db52921532a51c8db7/ROMS_tutorial.zip
unzip -oq ROMS_tutorial.zip

# ROMS source code, pinned to the version tested for this class
if [ ! -d roms ]; then
  git clone -q https://github.com/myroms/roms.git
  git -C roms checkout -q f8a3aaa205daaec17c52f028ba62961cc481d379
fi
echo "  ROMS version: $(git -C roms log -1 --format='%h  %cd' --date=short)"
ls /content/ROMS_tutorial/IO_1deg

In [ ]:
%cd /content/ROMS_tutorial/IO_1deg
APP = '/content/ROMS_tutorial/IO_1deg'

import numpy as np, xarray as xr, pandas as pd, matplotlib.pyplot as plt, re, os, shutil
import cartopy.crs as ccrs, cartopy.feature as cfeature
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': False})
PC = ccrs.PlateCarree()            # lon/lat data -> 'transform=PC' on every map plot

def map_axes(ncols=1, nrows=1, extent=None, figsize=None):
    # Figure with cartopy map axes: coastlines, land and labelled lon/lat gridlines
    fig, axs = plt.subplots(nrows, ncols, figsize=figsize or (7.5 * ncols, 4.6 * nrows),
                            subplot_kw={'projection': PC}, squeeze=False)
    for ax in axs.flat:
        if extent is not None:
            ax.set_extent(extent, crs=PC)
        ax.add_feature(cfeature.LAND.with_scale('50m'), facecolor='#d9d4c7', zorder=2)
        ax.coastlines('50m', linewidth=0.6, zorder=3)
        gl = ax.gridlines(crs=PC, draw_labels=True, linewidth=0.4, color='gray', alpha=0.6, linestyle='--')
        gl.top_labels = gl.right_labels = False
    return fig, (axs[0, 0] if axs.size == 1 else axs.ravel())

def on_rho(da, grid):
    # Attach 2-D lon/lat of the rho-points so xarray can plot them on the map
    return da.assign_coords(lon=(('eta_rho', 'xi_rho'), grid.lon_rho.values),
                            lat=(('eta_rho', 'xi_rho'), grid.lat_rho.values))

# ---- Which input files? ----------------------------------------------------------------
# 'mine'     : the files YOU made in notebook 1 (Grid_IC_BC_Preparation_Steps.ipynb), from Google Drive
# 'tutorial' : the ready-made grid / initial conditions / forcing shipped in the tutorial zip
INPUTS = 'mine'
MY_DIR = '/content/drive/MyDrive/ROMS_class/inputs'      # where notebook 1 saved your files

if INPUTS == 'mine':
    if not os.path.isdir(MY_DIR):
        from google.colab import drive
        drive.mount('/content/drive')
    needed = ['roms_grd.nc', 'roms_ini.nc', 'roms_frc.nc']
    missing = [f for f in needed if not os.path.exists(f'{MY_DIR}/{f}')]
    if missing:
        raise FileNotFoundError(f'{missing} not found in {MY_DIR}. Run notebook 1 to the end (its last cell '
                                f'saves to Drive), or set INPUTS = "tutorial" to use the ready-made files.')
    for f in needed:
        shutil.copy(f'{MY_DIR}/{f}', APP)
        print('  copied', f)
    GRD, INI, FRC = needed
else:
    GRD, INI, FRC = 'IO_1deg_grd.nc', 'IO_IC.nc', 'ERA5.nc'   # ERA5_midmonth.nc is made in Step 3
print('Using', GRD, INI, FRC, '| boundaries: closed walls')

## Step 2 — Look at the inputs before running anything

A model is only as good as its inputs. ROMS needs:

| File | Content | Used here? |
|---|---|---|
| `roms_grd.nc` (yours) · `IO_1deg_grd.nc` (tutorial) | grid: lon/lat, bathymetry `h`, land mask, Coriolis `f`, metrics `pm`,`pn` | ✅ |
| `roms_ini.nc` · `IO_IC.nc` | initial T, S, u, v, ζ on the model grid and s-levels | ✅ |
| `roms_frc.nc` · `ERA5.nc` | monthly atmosphere: winds, air T, humidity, pressure, radiation, rain | ✅ |
| `roms_bry.nc` | open-boundary values | ❌ boundaries are **closed walls** |

> 💡 Your files come from notebook 1 (`Grid_IC_BC_Preparation_Steps.ipynb`, made with *roms-tools*).

In [ ]:
grd = xr.open_dataset(f'{APP}/{GRD}')
ini = xr.open_dataset(f'{APP}/{INI}')
ocean = grd.mask_rho == 1
EXTENT = [float(grd.lon_rho.min()), float(grd.lon_rho.max()), float(grd.lat_rho.min()), float(grd.lat_rho.max())]

fig, ax = map_axes(2, extent=EXTENT)
on_rho(grd.h.where(ocean), grd).plot.pcolormesh(ax=ax[0], x='lon', y='lat', transform=PC, cmap='viridis_r',
                                               cbar_kwargs={'label': 'depth h (m)', 'shrink': 0.8})
ax[0].set_title('Bathymetry (smoothed for the 1° grid)')
sst_ini = ini.temp.isel(ocean_time=0, s_rho=-1).where(ocean)
on_rho(sst_ini, grd).plot.pcolormesh(ax=ax[1], x='lon', y='lat', transform=PC, cmap='RdYlBu_r',
                                     cbar_kwargs={'label': '°C', 'shrink': 0.8})
ax[1].set_title(f'Initial SST — {str(ini.ocean_time.values[0])[:10]}')

print('Grid size (xi_rho × eta_rho):', grd.sizes['xi_rho'], '×', grd.sizes['eta_rho'],
      '| grid spacing ≈ %.0f–%.0f km' % (1e-3/grd.pm.max(), 1e-3/grd.pm.min()))
print('s-coordinates in IC file: Vtransform=%d Vstretching=%d theta_s=%.1f theta_b=%.1f Tcline=%.0f m'
      % (ini.Vtransform, ini.Vstretching, ini.theta_s, ini.theta_b, ini.Tcline))

**Check:** the s-coordinate parameters in the initial-condition file must match `Vtransform`, `Vstretching`, `THETA_S`, `THETA_B` and `TCLINE` in `roms_io_1deg.in`. ROMS does **not** check this for you. If they don't match, temperature is silently placed at the wrong depths.

Next, the forcing. ERA5 here is **monthly means**: 12 records, one per month.

In [ ]:
if INPUTS == 'mine':                    # your forcing, already on the model grid
    frc = xr.open_dataset(f'{APP}/{FRC}')
    print('your forcing records:', frc.time.values.astype('datetime64[D]'))
    f1 = frc.sel(time='1993-01').isel(time=0)
    fig, ax = map_axes(3, extent=EXTENT, figsize=(19, 4.2))
    for a, (v, cm, lab) in zip(ax, [('ssrd', 'magma', 'net shortwave / 0.94 (W/m²)'),
                                    ('strd', 'magma', 'longwave DOWN (W/m²)'),
                                    ('t2m', 'RdYlBu_r', 'air temp (°C)')]):
        on_rho(f1[v], grd).plot.pcolormesh(ax=a, x='lon', y='lat', transform=PC, cmap=cm, robust=True,
                                           cbar_kwargs={'label': lab, 'shrink': 0.75})
        a.set_title(f'Your forcing: {v}, mid-Jan 1993')
    s = (slice(None, None, 4), slice(None, None, 4))
    ax[2].quiver(grd.lon_rho.values[s], grd.lat_rho.values[s], f1.u10.values[s], f1.v10.values[s],
                 transform=PC, scale=150, zorder=4)

# the original ERA5 file in the tutorial zip, for comparison
era = xr.open_dataset(f'{APP}/ERA5.nc')
print(era.time.values.astype('datetime64[D]'))
fig, ax = map_axes(3, extent=EXTENT, figsize=(19, 4.2))
m = era.sel(time='1993-01').isel(time=0)
for a, (v, cm, lab) in zip(ax, [('ssrd', 'magma', 'shortwave DOWN (W/m²)'),
                                ('strd', 'magma', 'longwave DOWN (W/m²)'),
                                ('t2m', 'RdYlBu_r', 'air temp (°C)')]):
    m[v].plot.pcolormesh(ax=a, x='lon', y='lat', transform=PC, cmap=cm, robust=True,   # robust: 2-98 % colour range
                         cbar_kwargs={'label': lab, 'shrink': 0.75})
    a.set_title(f'ERA5 {v}, Jan 1993')
w = m[['u10', 'v10']].isel(lon=slice(None, None, 3), lat=slice(None, None, 3))
ax[2].quiver(w.lon.values, w.lat.values, w.u10.values, w.v10.values, transform=PC, scale=150, zorder=4)

### 🧠 Two classic forcing traps, both present in last year's set-up

1. **Downward vs. net longwave.** ERA5 `strd` is the longwave radiation coming *down* from the sky, about 350–420 W/m² over the tropical ocean.
   By default ROMS expects *net* longwave (down − up ≈ −30 to −80 W/m²). Feeding `strd` in as "net" gives the ocean roughly +450 W/m² of spurious heating.
   **Fix:** `#define LONGWAVE_OUT`. ROMS then reads the *downward* flux and computes the outgoing part from the model's own SST (σT⁴).
2. **Monthly means stamped on the 1st.** A January mean represents mid-January. Stamping it on 1 January shifts the whole seasonal cycle half a month early.
   **Fix:** re-stamp the records to the 15th (done in Step 3).

Also worth knowing: ERA5 `ssrd` is *downward* shortwave. The ocean reflects about 6 % (albedo), so we scale it by 0.94.
`Q2M` is *specific* humidity. In `varinfo.yaml` it is multiplied by 1000 to give g/kg, which ROMS recognises because the values are ≥ 2. Values < 2 would be read as relative humidity (a fraction).

## Step 3 — Configure the model

ROMS is configured in **three places**:

| File | Controls | When it takes effect |
|---|---|---|
| `io_1deg.h` | **which physics/numerics are compiled in** (CPP `#define` switches) | needs **re-compiling** |
| `roms_io_1deg.in` | run length, time step, viscosity, output frequency, file names | read at run time |
| `varinfo.yaml` | names, units and scale factors of every variable ROMS reads or writes | read at run time |

### 3a. Physics switches — `io_1deg.h`
Read the comments: each line is a modelling decision.

In [ ]:
%%writefile io_1deg.h
/*
** Options for the 1-degree Indian Ocean teaching configuration.
** Application flag: IO_1DEG        Input script: roms_io_1deg.in
** All CPP options live in THIS file (build_roms.sh adds none).
*/

/* ---- Momentum ------------------------------------------------------- */
#define UV_ADV            /* advection of momentum                        */
#define UV_COR            /* Coriolis                                     */
#define UV_QDRAG          /* quadratic bottom drag (RDRG2 in .in)         */
#define UV_VIS2           /* Laplacian viscosity (VISC2 in .in)           */
#define MIX_S_UV          /* ...applied along s-surfaces                  */

/* ---- Tracers -------------------------------------------------------- */
#define SALINITY
#define NONLIN_EOS        /* nonlinear equation of state                  */
#define TS_DIF2           /* Laplacian tracer diffusion (TNU2 in .in)     */
#define MIX_GEO_TS        /* ...along geopotential surfaces: avoids       */
                          /* spurious mixing over steep s-coordinates     */
#define SOLAR_SOURCE      /* shortwave penetrates below the surface       */

/* ---- Grid & numerics ------------------------------------------------ */
#define SOLVE3D
#define CURVGRID
#define MASKING
#define DJ_GRADPS         /* pressure-gradient scheme                     */
#define SPLINES_VDIFF
#define SPLINES_VVISC

/* ---- Vertical mixing: KPP ------------------------------------------- */
/* Without a scheme, ROMS only has the tiny background AKT_BAK (5e-6):   */
/* no mixed layer, no convection.                                        */
#define LMD_MIXING
#ifdef LMD_MIXING
# define LMD_RIMIX        /* shear mixing in the interior                 */
# define LMD_CONVEC       /* convection when the column is unstable       */
# define LMD_SKPP         /* surface boundary layer                       */
# define LMD_BKPP         /* bottom boundary layer                        */
# define LMD_NONLOCAL
# define RI_SPLINES
#endif

/* ---- Surface forcing: bulk formulae with ERA5 ----------------------- */
#define BULK_FLUXES
#ifdef BULK_FLUXES
# define LONGWAVE_OUT     /* input = DOWNWARD longwave (ERA5 strd)        */
# define EMINUSP          /* salt flux from evaporation - precipitation   */
#endif

/* ---- Bottom tracer fluxes = 0 --------------------------------------- */
#define ANA_BSFLUX
#define ANA_BTFLUX

/* ---- Output --------------------------------------------------------- */
#define AVERAGES

### Step 3b — Compile

`build_roms.sh` sets the compiler (`gfortran`), MPI and NetCDF options, then calls ROMS's `make`.
It uses `ROMS_ROOT_DIR` to find the source code we cloned in `/content/roms`.

The compiler output runs to thousands of lines, so we send it to `build.log` and show only the end.
Success means a `romsM` executable appears in the folder.

In [ ]:
# --- build_roms.sh: remove duplicated CPP flags (they are now all in io_1deg.h)
b = open('build_roms.sh').read()
b = re.sub(r'^ export(\s+MY_CPP_FLAGS="\$\{MY_CPP_FLAGS\} -D(AVERAGES|DIAGNOSTICS_TS|DIAGNOSTICS_UV|TS_DIF2|MIX_S_TS|UV_VIS2|MIX_S_UV)")',
           r'#export\1', b, flags=re.M)
open('build_roms.sh', 'w').write(b)
print('build_roms.sh: extra CPP flags commented out')

In [ ]:
%%shell
export ROMS_ROOT_DIR=/content
cd /content/ROMS_tutorial/IO_1deg
time ./build_roms.sh -j 2 > build.log 2>&1
grep -iE "^.*error" build.log | head -5
tail -6 build.log
ls -l romsM && echo "✅ ROMS compiled" || echo "❌ build failed: inspect build.log"

## 4. Run parameters, file names and the forcing fix

The helper below edits `KEY == value` lines in the `.in` file, so you can see exactly which parameters change.

**Key choices:**
* `DT = 3000 s`, so one model day = 28.8 steps.
* `VISC2 = 1e4 m²/s`. On a 1° grid, the western-boundary (Munk) layer, (A/β)^⅓, must be at least one grid cell wide, which needs A ≈ 10⁴ m²/s.
  Last year's value of 50 m²/s left the model nearly inviscid: grid-scale noise and currents of **4–10 m/s** off Somalia and in the Gulf of Aden.
* Output: 5-day **averages**, 10-day **history** snapshots, and a **restart** file at the end of the run.
* `DSTART` and `TIME_REF`: ROMS counts time steps from `TIME_REF`, so `DSTART` must equal the initial-condition time. Last year's value was `DSTART = 0`, which offset every output window; the first 5-day average then came out about 5 °C too cold.

In [ ]:
RUN_DAYS = 30                     # ← change me (each model day ≈ 5–10 s on Colab)
DT = 3000.0                       # baroclinic time step (s)
steps_per_day = 86400 / DT

def set_param(fname, key, value):
    '''Replace the value of an active "KEY == value" line in a ROMS .in file.'''
    txt = open(fname).read()
    pat = re.compile(r'^(\s*' + re.escape(key) + r'\s*==?)\s.*$', re.M)
    if len(pat.findall(txt)) != 1:
        raise ValueError(f'{key}: found {len(pat.findall(txt))} active lines')
    open(fname, 'w').write(pat.sub(lambda m: f'{m.group(1)} {value}', txt))
    print(f'  {key:14s} = {value}')

# ROMS counts time steps from TIME_REF. DSTART must equal the time of the
# initial-condition file, otherwise output windows don't line up with the start
# and the first 5-day average is wrong (last year's run had this problem).
ic = xr.open_dataset(INI, decode_times=False)
DSTART = float(ic.ocean_time[0]) / 86400.0          # IC time: seconds since 1970-01-01 -> days

# Grid size and s-coordinates must match the input files exactly -> read them from the files
gsz = xr.open_dataset(GRD).sizes
VERT = {'Lm': gsz['xi_rho'] - 2, 'Mm': gsz['eta_rho'] - 2, 'N': ic.sizes['s_rho'],
        'Vtransform': int(ic.Vtransform), 'Vstretching': int(ic.Vstretching),
        'THETA_S': f'{float(ic.theta_s)}d0', 'THETA_B': f'{float(ic.theta_b)}d0', 'TCLINE': f'{float(ic.Tcline)}d0'}

IN = 'roms_io_1deg.in'
print('Editing', IN)
for key, val in {
        'TIME_REF': '19700101.0d0',      # reference date for all model times
        'DSTART': f'{DSTART:.10f}d0',    # start = IC time, in days since TIME_REF
        'NTIMES': int(RUN_DAYS * steps_per_day),   # number of time steps
        'DT':     '3000.0d0',
        'NINFO':  72,                    # print diagnostics every 2.5 days (not every step!)
        'NAVG':   int(5 * steps_per_day),    # 5-day averages
        'NHIS':   int(10 * steps_per_day),   # 10-day snapshots
        'NRST':   int(RUN_DAYS * steps_per_day),  # restart at the end
        'VISC2':  '1.0d4',               # m2/s  harmonic viscosity
        'TNU2':   '1000.0d0 1000.0d0',    # m2/s  harmonic diffusivity (T, S)
        **VERT,
        'GRDNAME': f'{APP}/{GRD}',
        'ININAME': f'{APP}/{INI}',
        'FRCNAME': f'{APP}/{FRC}',
        'NBCFILES': 1,
        'BRYNAME': 'roms_bry.nc',                     # not read: all edges are closed
        # nudging time scale toward boundary data (days); inflow is OBCFAC = 3 times stronger
        'TNUDG': '10.0d0 10.0d0', 'ZNUDG': '10.0d0', 'M2NUDG': '10.0d0', 'M3NUDG': '10.0d0',
        # extra fields written to the averages file (used in Step 7)
        'Aout(idUsms)': 'T', 'Aout(idVsms)': 'T', 'Aout(idTsur)': 'T T',
        'Aout(idSrad)': 'T', 'Aout(idLrad)': 'T', 'Aout(idLhea)': 'T',
        'Aout(idShea)': 'T', 'Aout(idHsbl)': 'T',
        }.items():
    set_param(IN, key, val)

# ---- Lateral boundary conditions: CLOSED walls on all four sides (west, south, east, north) ----
txt = open(IN).read()
for k in ['isFsur', 'isUbar', 'isVbar', 'isUvel', 'isVvel', 'isMtke']:
    txt, n = re.subn(r'^(\s*LBC\(' + k + r'\)\s*==).*$', lambda m: f'{m.group(1)}   Clo Clo Clo Clo', txt, flags=re.M)
    assert n == 1, k
# the tracer line continues on the next line (temperature \ salinity)
txt, n = re.subn(r'^(\s*LBC\(isTvar\)\s*==).*\n.*$',
                 lambda m: f'{m.group(1)}   Clo Clo Clo Clo \\\n                    Clo Clo Clo Clo', txt, flags=re.M)
assert n == 1
open(IN, 'w').write(txt)

# check: print the boundary lines ROMS will read
lbc = re.findall(r'^\s*LBC\(is\w+\)\s*==.*(?:\\\n.*)?', txt, flags=re.M)
for l in lbc:
    print(' ', ' '.join(l.split('!')[0].split()))
assert set(re.findall(r'\b(Clo|Rad|RadNud|Cha|Fla|Gra|Nes|Per|Red|Shc)\b', ' '.join(lbc))) == {'Clo'}
print('  ✅ all lateral boundaries are closed walls: no boundary file is read')

In [ ]:
# --- varinfo.yaml: tell ROMS which ERA5 variable is which -----------------
vi = open('varinfo.yaml').read()
fixes = [
  # (a) downward longwave: read ERA5 'strd' into idLdwn (used with LONGWAVE_OUT)
  ("  - variable:       lwrad_down                                       # Input",
   "  - variable:       strd                                             # Input (ERA5 downward LW)"),
  ("    field:          downwelling longwave radiation                   # [Celsius m/s]\n    time:           lrf_time",
   "    field:          downwelling longwave radiation                   # [Celsius m/s]\n    time:           time"),
  # (b) the NET longwave entry is now only an OUTPUT, so give it its own name
  ("  - variable:       strd                                            # Input/Output",
   "  - variable:       lwrad                                           # Output (computed net LW)"),
  # (c) shortwave: ERA5 ssrd is downward; ocean albedo ~6 %
  ("    index_code:     idSrad\n    type:           r2dvar\n    add_offset:     0.0d0\n    scale:          1.0d0",
   "    index_code:     idSrad\n    type:           r2dvar\n    add_offset:     0.0d0\n    scale:          0.94d0"),
]
for old, new in fixes:
    if new in vi:   continue                     # already applied (safe to re-run)
    assert vi.count(old) == 1, 'pattern not found: ' + old[:50]
    vi = vi.replace(old, new)
open('varinfo.yaml', 'w').write(vi)
print('varinfo.yaml updated')

# --- Forcing: re-stamp monthly means to mid-month ---------------------------
# Add Dec-1992 (copy of Dec-1993) and Jan-1994 (copy of Jan-1993) so a run
# starting on 15 Jan can interpolate between records.
# e = xr.open_dataset('ERA5.nc').drop_vars('time_bnds')
# mid = pd.to_datetime(e.time.values) + pd.Timedelta(days=14)
# out = xr.concat([e.isel(time=[-1]).assign_coords(time=[pd.Timestamp('1992-12-15')]),
#                  e.assign_coords(time=mid),
#                  e.isel(time=[0]).assign_coords(time=[pd.Timestamp('1994-01-15')])], 'time')
# out.time.encoding.update(units='days since 1970-01-01', calendar='gregorian', dtype='float64')
# # Write classic NetCDF-3 with scipy (always installed on Colab). NetCDF-3 stores
# # text attributes as plain characters, which ROMS (Fortran) can read. Colab's
# # default writer (h5netcdf) uses NC_STRING attributes instead, which makes ROMS stop with
# # "NETCDF_GET_SATT - error while reading attribute: units".
# out.to_netcdf('ERA5_midmonth.nc', format='NETCDF3_64BIT', engine='scipy')
# print('ERA5_midmonth.nc written:', out.time.values[[0, 1, -1]].astype('datetime64[D]'))

<details><summary>💬 <b>Messages you can ignore</b> (click)</summary>

* `debconf`, `ldconfig ... is not a symbolic link`: Colab's apt noise.
* `Warning: Rank mismatch` / `Type mismatch` in MPI routines: gfortran is being strict about old-style MPI interfaces. Harmless.
* `"UV_VIS2" redefined`: means a flag is set both in `build_roms.sh` and in the header. We removed the duplicates in Step 3.
</details>

## Step 5 — Run

The log goes to `roms.log`. While the model runs, you'll see one line each time a 5-day average is written.

Each diagnostic block in the log reports:
`STEP  date  KINETIC_ENRG  POTEN_ENRG  TOTAL_ENRG  NET_VOLUME` and on the next line `(i,j,k)  Cu  Cv  Cw  Max Speed`.

> ⏱ Expect about 5–10 s per model day on a free Colab CPU (30 days ≈ 3–5 min; tested at 4.4 s/day on 2 cores). If time is short, set `RUN_DAYS = 10` in Step 3, re-run Step 3, and run this cell again (no need to recompile).
> ROMS also runs in parallel. Set `NtileI == 2` in the `.in` file and use `mpirun --allow-run-as-root -np 2 ./romsM roms_io_1deg.in`. Free Colab has only two cores, so the gain is small.

In [ ]:
%%shell
cd /content/ROMS_tutorial/IO_1deg
rm -f roms_his.nc roms_avg.nc roms_rst.nc
time ./romsM roms_io_1deg.in | tee roms.log | grep --line-buffered -E "WRT_AVG|Blowing-up|ERROR|Found Error|cannot find|ROMS: DONE"
echo "------------------------------------------------------------"
if grep -q "Output NetCDF summary" roms.log; then
  grep -A12 "Output NetCDF summary" roms.log | grep -E "records|DONE"
else
  echo "❌ ROMS stopped early. Last lines of roms.log:"
  grep -B2 -A4 -E "Found Error|ERROR|cannot" roms.log | head -40
  tail -15 roms.log
fi
true

### Is the run healthy? Read the log like a modeller

The two quickest stability checks are **domain kinetic energy** (should settle, not grow without bound) and **maximum speed** (should be physically plausible: at 1°, about 1–1.5 m/s at most, in the Somali Current).

In [ ]:
def read_diag(logfile):
    L = open(logfile).read().splitlines(); rows = []
    for i, l in enumerate(L[:-1]):
        m = re.match(r'\s+\d+ (\d{4}-\d\d-\d\d \S+)\s+(\S+)\s+\S+\s+\S+\s+\S+$', l)
        p = L[i+1].split()
        if m and len(p) == 5 and p[0].startswith('('):
            rows.append(dict(time=pd.to_datetime(m.group(1)[:19]), KE=float(m.group(2)),
                             where=p[0], Cw=float(p[3]), maxspeed=float(p[4])))
    if not rows:
        raise RuntimeError('No time steps in roms.log: the run failed. Look for "ERROR" or "cannot find" in roms.log')
    return pd.DataFrame(rows).set_index('time')

d = read_diag('roms.log')
fig, ax = plt.subplots(1, 2, figsize=(12, 3))
d.KE.plot(ax=ax[0], marker='.', title='Domain-mean kinetic energy (m²/s²)')
d.maxspeed.plot(ax=ax[1], marker='.', color='C3', title='Maximum speed (m/s)')
plt.tight_layout()
print(d.tail(3))

> 🔬 **Exercise (after class):** in Step 3 set `VISC2` back to `50.0d0`, re-run Steps 3 and 5 (no recompile needed), and compare the max-speed plot. Last year's configuration reached 4 m/s on the first day.

## Step 6 — Save your output

Colab deletes everything when the runtime disconnects. Download the files or copy them to Google Drive.

In [ ]:
# Option A: download to your computer
# from google.colab import files
# files.download('roms_avg.nc')

# Option B: copy to Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# !mkdir -p /content/drive/MyDrive/ROMS_class && cp roms_*.nc roms.log /content/drive/MyDrive/ROMS_class/
!ls -lh roms_*.nc

## Step 7 — Analyse the output

| File | Content |
|---|---|
| `roms_avg.nc` | 5-day **averages** of u, v, T, S, ζ, surface fluxes, mixed-layer depth |
| `roms_his.nc` | **snapshots** every 10 days |
| `roms_rst.nc` | restart file: continue the run later from here |

### 7a. Open the file and understand the C-grid
ROMS uses an **Arakawa C-grid**: T, S and ζ sit at cell centres (ρ-points), `u` sits on the east/west faces and `v` on the north/south faces.
That's why `u` has dimensions `(eta_u, xi_u)` and is one point shorter in x.

In [ ]:
avg = xr.open_dataset('roms_avg.nc')
his = xr.open_dataset('roms_his.nc')
print('averages:', avg.ocean_time.values.astype('datetime64[D]'))
print('dims of temp:', avg.temp.dims, '| u:', avg.u.dims, '| v:', avg.v.dims)
mask = avg.mask_rho.where(avg.mask_rho == 1)       # 1 over ocean, NaN over land
lon, lat = avg.lon_rho, avg.lat_rho

### 7b. Sea-surface temperature and its change since the start

In [ ]:
sst0 = his.temp.isel(ocean_time=0, s_rho=-1) * mask
sst  = avg.temp.isel(ocean_time=-1, s_rho=-1) * mask
fig, ax = map_axes(2, extent=EXTENT)
on_rho(sst, avg).plot.pcolormesh(ax=ax[0], x='lon', y='lat', transform=PC, cmap='RdYlBu_r', vmin=18, vmax=31,
                                 cbar_kwargs={'label': '°C', 'shrink': 0.8})
ax[0].set_title(f'SST, 5-day mean ending {str(avg.ocean_time.values[-1])[:10]}')
on_rho(sst - sst0, avg).plot.pcolormesh(ax=ax[1], x='lon', y='lat', transform=PC, cmap='RdBu_r', vmin=-3, vmax=3,
                                        cbar_kwargs={'label': '°C', 'shrink': 0.8})
ax[1].set_title('SST change since initial state')

### 7c. Surface currents (move u, v to ρ-points first)

In [ ]:
def uv_to_rho(u, v, angle):
    '''Average C-grid u, v to interior rho-points and rotate to east/north.'''
    ur = 0.5 * (u[..., :, :-1].values + u[..., :, 1:].values)[..., 1:-1, :]
    vr = 0.5 * (v[..., :-1, :].values + v[..., 1:, :].values)[..., :, 1:-1]
    a = angle.values[1:-1, 1:-1]
    return ur*np.cos(a) - vr*np.sin(a), vr*np.cos(a) + ur*np.sin(a)

ue, vn = uv_to_rho(avg.u.isel(ocean_time=-1, s_rho=-1), avg.v.isel(ocean_time=-1, s_rho=-1), avg.angle)
X, Y, M = lon.values[1:-1, 1:-1], lat.values[1:-1, 1:-1], mask.values[1:-1, 1:-1]
spd = np.hypot(ue, vn) * M

fig, ax = map_axes(extent=EXTENT, figsize=(11, 6))
p = ax.pcolormesh(X, Y, spd, transform=PC, cmap='viridis', vmax=1.0, shading='auto')
s = (slice(None, None, 2), slice(None, None, 2))
q = ax.quiver(X[s], Y[s], (ue*M)[s], (vn*M)[s], transform=PC, scale=8, width=0.0025, zorder=4)
ax.quiverkey(q, 0.9, -0.12, 0.5, '0.5 m/s', labelpos='E')
plt.colorbar(p, ax=ax, label='speed (m/s)', shrink=0.8)
ax.set_title(f'Surface currents, {str(avg.ocean_time.values[-1])[:10]}  (max {np.nanmax(spd):.2f} m/s)')

💬 *Discussion:* January–February is the NE (winter) monsoon. Look for a **southward Somali Current**, the **South Equatorial Current** near 10°S and the **East African Coastal Current**.
At 1° resolution, how many grid cells wide is the Somali Current?

### 7d. From s-levels to real depths, and a vertical section

ROMS levels follow the terrain. To plot against depth we compute `z` for every point (here `Vtransform = 2`):

$$z = \zeta + (\zeta + h)\,\frac{h_c\,s + h\,C_s}{h_c + h}$$

In [ ]:
def z_rho(ds, t=-1):
    h, hc, zeta = ds.h, float(ds.hc), ds.zeta.isel(ocean_time=t)
    z0 = (hc * ds.s_rho + h * ds.Cs_r) / (hc + h)
    return (zeta + (zeta + h) * z0).transpose('s_rho', 'eta_rho', 'xi_rho')

z = z_rho(avg)
j = int(abs(lat[:, 0] - 0.0).argmin())        # row closest to the equator (try 10 for 10°N!)
T = avg.temp.isel(ocean_time=-1, eta_rho=j).where(avg.mask_rho.isel(eta_rho=j) == 1)
Xs = np.broadcast_to(lon.isel(eta_rho=j).values, T.shape)

fig = plt.figure(figsize=(16, 4.2))
axm = fig.add_subplot(1, 3, 1, projection=PC)                  # locator map (cartopy)
axm.set_extent(EXTENT, crs=PC)
axm.add_feature(cfeature.LAND.with_scale('50m'), facecolor='#d9d4c7'); axm.coastlines('50m', linewidth=0.6)
axm.plot(lon.isel(eta_rho=j), lat.isel(eta_rho=j), 'r-', lw=2.5, transform=PC)
axm.set_title('Section location')
ax = fig.add_subplot(1, 3, (2, 3))                             # the section itself
p = ax.contourf(Xs, z.isel(eta_rho=j), T, levels=np.arange(4, 31, 1), cmap='RdYlBu_r', extend='both')
ax.contour(Xs, z.isel(eta_rho=j), T, levels=[20], colors='k', linewidths=1.5)   # 20°C ≈ thermocline
ax.fill_between(Xs[0], -avg.h.isel(eta_rho=j), -500, color='#d9d4c7')
ax.set_ylim(-500, 0); plt.colorbar(p, ax=ax, label='°C')
ax.set_title(f'Temperature along {float(lat[j,0]):.1f}°N (black: 20 °C isotherm ≈ thermocline)')
ax.set_xlabel('Longitude'); ax.set_ylabel('Depth (m)')

### 7e. Mixed-layer depth from KPP

`Hsbl` is the depth of the KPP surface boundary layer. Winter cooling and NE monsoon winds deepen it in the northern Arabian Sea.
Try commenting out `LMD_MIXING` in `io_1deg.h`, recompiling and re-running: the mixed layer disappears entirely.

In [ ]:
fig, ax = map_axes(extent=EXTENT, figsize=(10, 5.5))
# ROMS stores Hsbl as a negative depth (z, positive up) -> flip the sign
mld = -avg.Hsbl.isel(ocean_time=-1) * mask
on_rho(mld, avg).plot.pcolormesh(ax=ax, x='lon', y='lat', transform=PC, cmap='cividis_r', vmin=0, vmax=100,
                                 cbar_kwargs={'label': 'm', 'shrink': 0.8})
ax.set_title('KPP surface boundary-layer depth (Hsbl)')

### 7f. Surface heat budget

Averaged over the domain, net longwave should be **negative** (the ocean loses heat to the sky).
If it shows hundreds of W/m², the downward/net mix-up from Step 2 is back.

In [ ]:
# area-weighted ocean mean
area = (1 / avg.pm / avg.pn) * avg.mask_rho
amean = lambda f: ((f * area).sum(('eta_rho', 'xi_rho')) / area.sum()).rename(f.name)
budget = pd.DataFrame({name: amean(avg[v]).to_series() for name, v in
          [('shortwave', 'ssrd'), ('longwave', 'lwrad'), ('latent', 'latent'),
           ('sensible', 'sensible'), ('NET', 'shflux')]})
budget.index = budget.index.strftime('%d-%b')
ax = budget.plot(kind='bar', figsize=(10, 3.5), rot=0, title='Domain-mean surface heat fluxes (W/m², + = into ocean)')
ax.axhline(0, color='k', lw=0.8); print(budget.round(1))

### 7g. Spin-up / drift: domain-mean SST and kinetic energy

In [ ]:
sst_ts = amean(avg.temp.isel(s_rho=-1))
ke = 0.5 * (avg.u.isel(s_rho=-1)**2).mean(('eta_u', 'xi_u')) + 0.5 * (avg.v.isel(s_rho=-1)**2).mean(('eta_v', 'xi_v'))
fig, ax = plt.subplots(1, 2, figsize=(12, 3))
ax[0].plot(sst_ts.ocean_time, sst_ts, 'o-'); ax[0].set_title('Domain-mean SST (°C)')
ax[1].plot(ke.ocean_time, ke, 'o-', color='C2'); ax[1].set_title('Mean surface KE (m²/s²)')
for a in ax: a.tick_params(axis='x', rotation=30)
plt.tight_layout()

### 7h. Model vs. reanalysis: GLORYS January 1993

`GLORY_Jan_1deg.nc` is the January-1993 mean from the GLORYS reanalysis. The model starts from it, so after a month the difference shows how far the free-running model **drifts**.

In [ ]:
gl = xr.open_dataset('/content/ROMS_tutorial/GLORY_Jan_1deg.nc').isel(time=0, depth=0)
gl_on_model = gl.thetao.interp(longitude=lon, latitude=lat)
first_avg = avg.temp.isel(ocean_time=0, s_rho=-1) * mask       # first 5-day mean (≈ late Jan)
diff = first_avg - gl_on_model
fig, ax = map_axes(extent=EXTENT, figsize=(10, 5.5))
on_rho(diff, avg).plot.pcolormesh(ax=ax, x='lon', y='lat', transform=PC, cmap='RdBu_r', vmin=-2, vmax=2,
                                  cbar_kwargs={'label': '°C', 'shrink': 0.8})
ax.set_title(f'ROMS − GLORYS SST (Jan), mean bias {float(diff.mean()):+.2f} °C, RMS {float(np.sqrt((diff**2).mean())):.2f} °C')

## Step 8 — Exercises

1. **Viscosity:** set `VISC2 = 50.0d0` and compare the max speed and the current field (Step 5 and 7c).
2. **Mixing:** comment out `LMD_MIXING`, recompile, re-run. What happens to SST and `Hsbl`?
3. **Longwave trap:** comment out `LONGWAVE_OUT` *and* map `strd` back to the net-longwave entry. Plot the heat budget (7f). How fast does SST rise?
4. **Restart:** continue the run for 30 more days from `roms_rst.nc`. Set `NRREC == -1` and `ININAME == roms_rst.nc`.
5. **Section:** draw a temperature section along 10°N in the Arabian Sea and in the Bay of Bengal. Where is the thermocline shallower?

## Appendix — How the model was set up
* **Grid / initial conditions:** see `Grid_Preparation.ipynb` (roms-tools + SRTM15 + GLORYS).
* **ROMS documentation:** https://www.myroms.org/wiki/ (CPP options: `ROMS/External/varinfo.yaml`, `ROMS/Include/cppdefs.h`).